In [1]:
import os
import glob
import numpy as np
import math
from scipy.spatial import cKDTree
from tqdm import tqdm

DATASET_PATH = r"Novi Dataset\npy"
OUTPUT_PATH  = r"Novi Dataset\patches_8000"
TARGET       = 8000
NUM_CLASSES  = 4

os.makedirs(OUTPUT_PATH, exist_ok=True)
print(f"Greedy patch rezanje | target: {TARGET} tocaka | izlaz: {OUTPUT_PATH}")

Greedy patch rezanje | target: 8000 tocaka | izlaz: Novi Dataset\patches_8000


In [2]:
def greedy_patchevi(points, target=TARGET):
    """
    Greedy region growing: kreni od najnize preostale tocke, uzmi target najblizih.
    Deterministicki. Svaki patch tocno target (zadnji dopunjen preklapanjem).
    Vraca listu nizova ORIGINALNIH indeksa (za rekonstrukciju).
    """
    N = points.shape[0]
    preostalo = np.ones(N, dtype=bool)
    tree_full = cKDTree(points)
    patchevi = []

    while preostalo.sum() > 0:
        preostali_idx = np.where(preostalo)[0]

        if len(preostali_idx) >= target:
            pts_pre = points[preostali_idx]
            sjeme_local = np.lexsort((pts_pre[:, 0], pts_pre[:, 1], pts_pre[:, 2]))[0]
            sjeme = preostali_idx[sjeme_local]

            tree_pre = cKDTree(points[preostali_idx])
            _, lokalni = tree_pre.query(points[sjeme], k=target)
            patch = preostali_idx[np.atleast_1d(lokalni)]

            patchevi.append(patch)
            preostalo[patch] = False
        else:
            idx = preostali_idx
            fali = target - len(idx)
            centar = points[idx].mean(axis=0)
            k_trazi = min(N, len(idx) + fali * 3 + 10)
            _, kand = tree_full.query(centar, k=k_trazi)
            kand = np.atleast_1d(kand)
            u_patchu = set(idx.tolist())
            vanjski = [c for c in kand if c not in u_patchu][:fali]
            patch = np.concatenate([idx, np.array(vanjski, dtype=idx.dtype)])
            patchevi.append(patch)
            preostalo[idx] = False

    return patchevi

In [3]:
npy_pattern = os.path.join(DATASET_PATH, "tree_*", "*.npy")
all_npy_files = glob.glob(npy_pattern)
print(f"Found {len(all_npy_files)} files.")

Found 657 files.


In [4]:
total_patches = 0
patch_counts = []
preklapanja = []

for fp in tqdm(all_npy_files, desc="Greedy patch rezanje"):
    tree = os.path.basename(os.path.dirname(fp))
    cloud_name = os.path.splitext(os.path.basename(fp))[0]

    d = np.load(fp, allow_pickle=True).item()
    pts = np.array(d['points'].T, dtype=np.float32)     # (N, 3)
    cols = np.array(d['colors'].T, dtype=np.float32)    # (N, 3), 0-255
    lbls = np.array(d['labels'], dtype=np.int64)        # (N,)

    patchevi = greedy_patchevi(pts, TARGET)

    out_dir = os.path.join(OUTPUT_PATH, tree, cloud_name)
    os.makedirs(out_dir, exist_ok=True)

    for i, orig_idx in enumerate(patchevi):
        out_fp = os.path.join(out_dir, f"{cloud_name}_{i}.npy")
        np.save(out_fp, {
            'points': pts[orig_idx],
            'colors': cols[orig_idx],
            'labels': lbls[orig_idx],
            'orig_idx': orig_idx.astype(np.int64),   # KLJUCNO za rekonstrukciju
            'tree_id': tree,
            'cloud_id': cloud_name,
            'patch_id': i,
            'n_original': pts.shape[0],               # koliko tocaka ima cijeli oblak
        })

    ukupno = sum(len(p) for p in patchevi)
    preklapanja.append(ukupno - pts.shape[0])
    total_patches += len(patchevi)
    patch_counts.append(len(patchevi))

patch_counts = np.array(patch_counts)
preklapanja = np.array(preklapanja)

print(f"\nGotovo. Ukupno patcheva: {total_patches}")
print(f"Patcheva po oblaku -> min: {patch_counts.min()} | max: {patch_counts.max()} | prosjek: {patch_counts.mean():.1f}")
print(f"Preklapanje -> prosjek: {preklapanja.mean():.0f} tocaka po oblaku")

Greedy patch rezanje: 100%|██████████| 657/657 [03:20<00:00,  3.27it/s]


Gotovo. Ukupno patcheva: 8107
Patcheva po oblaku -> min: 3 | max: 23 | prosjek: 12.3
Preklapanje -> prosjek: 4098 tocaka po oblaku


In [6]:
import numpy as np, glob, os
import open3d as o3d

tree = "tree_1_V_0000"
cloud = "0"

# --- ORIGINAL ---
orig_fp = os.path.join(r"Novi Dataset\npy", tree, f"{cloud}.npy")
d = np.load(orig_fp, allow_pickle=True).item()
orig_pts = np.array(d['points'].T, dtype=np.float64)
orig_cols = np.array(d['colors'].T, dtype=np.float64)
orig_cols = orig_cols / 255.0 if orig_cols.max() > 1.0 else orig_cols
N = orig_pts.shape[0]
print(f"ORIGINAL: {N} tocaka")

# --- REKONSTRUKCIJA iz patcheva (preko orig_idx) ---
patch_dir = os.path.join(r"Novi Dataset\patches_8000", tree, cloud)
patch_files = glob.glob(os.path.join(patch_dir, "*.npy"))
print(f"Patcheva: {len(patch_files)}")

# rekonstruiraj: svaka originalna tocka dobije svoje podatke natrag preko orig_idx
rek_pts = np.zeros((N, 3))
rek_cols = np.zeros((N, 3))
pokriveno = np.zeros(N, dtype=bool)

ukupno_patch_tocaka = 0
for fp in patch_files:
    p = np.load(fp, allow_pickle=True).item()
    oi = p['orig_idx']
    rek_pts[oi] = p['points']
    c = p['colors'].astype(np.float64)
    c = c / 255.0 if c.max() > 1.0 else c
    rek_cols[oi] = c
    pokriveno[oi] = True
    ukupno_patch_tocaka += len(oi)

print(f"\nUkupno tocaka u svim patchevima (s preklapanjem): {ukupno_patch_tocaka}")
print(f"Jedinstvenih pokrivenih (rekonstruirano): {pokriveno.sum()} / {N}")
print(f"Sve pokriveno? {'DA' if pokriveno.all() else 'NE - fali ' + str(N - pokriveno.sum())}")
print(f"Koordinate identicne originalu? {np.allclose(rek_pts, orig_pts)}")

# --- vizualizacija: original i rekonstruirano ---
pcd_orig = o3d.geometry.PointCloud()
pcd_orig.points = o3d.utility.Vector3dVector(orig_pts)
pcd_orig.colors = o3d.utility.Vector3dVector(orig_cols)

pcd_rek = o3d.geometry.PointCloud()
pcd_rek.points = o3d.utility.Vector3dVector(rek_pts)
pcd_rek.colors = o3d.utility.Vector3dVector(rek_cols)

o3d.visualization.draw_geometries([pcd_orig], window_name=f"ORIGINAL | {N} tocaka")
o3d.visualization.draw_geometries([pcd_rek],  window_name=f"REKONSTRUIRANO | {pokriveno.sum()} tocaka")

ORIGINAL: 128919 tocaka
Patcheva: 17

Ukupno tocaka u svim patchevima (s preklapanjem): 136000
Jedinstvenih pokrivenih (rekonstruirano): 128919 / 128919
Sve pokriveno? DA
Koordinate identicne originalu? True


In [7]:
import numpy as np, glob, os
import open3d as o3d

tree = "tree_1_V_0000"
cloud = "0"

# --- ucitaj originalni oblak ---
orig_fp = os.path.join(r"Novi Dataset\npy", tree, f"{cloud}.npy")
d = np.load(orig_fp, allow_pickle=True).item()
points = np.array(d['points'].T, dtype=np.float64)
N = points.shape[0]
print(f"Oblak: {tree}/{cloud} | {N} tocaka")

# --- ucitaj patcheve tog oblaka ---
patch_dir = os.path.join(r"Novi Dataset\patches_8000", tree, cloud)
patch_files = sorted(
    glob.glob(os.path.join(patch_dir, "*.npy")),
    key=lambda f: int(os.path.splitext(os.path.basename(f))[0].split("_")[-1])
)
print(f"Patcheva: {len(patch_files)} (svaki 8000 tocaka)")

# --- oboji svaki patch svojom bojom (preko orig_idx) ---
rng = np.random.default_rng(42)
boje = np.zeros((N, 3))
geometrije = []

for fp in patch_files:
    p = np.load(fp, allow_pickle=True).item()
    oi = p['orig_idx']
    b = rng.random(3)
    boje[oi] = b

    # bounding box patcha
    pts_p = points[oi]
    aabb = o3d.geometry.AxisAlignedBoundingBox(pts_p.min(axis=0), pts_p.max(axis=0))
    aabb.color = b
    geometrije.append(aabb)

pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(points)
pcd.colors = o3d.utility.Vector3dVector(boje)
geometrije.append(pcd)

# --- 2 prikaza: samo tocke (cisto), pa tocke + okviri ---
o3d.visualization.draw_geometries([pcd],
    window_name=f"{tree}/{cloud} — {len(patch_files)} patcheva x 8000 tocaka")

o3d.visualization.draw_geometries(geometrije,
    window_name=f"{tree}/{cloud} — patchevi + granice")

Oblak: tree_1_V_0000/0 | 128919 tocaka
Patcheva: 17 (svaki 8000 tocaka)
